# 12.4 時間框的邊緣如何處理？


剪下一小段波形，起點與終點可能不接在同一個高度。如果頻率分析把這段當成可以首尾重複的片段，接縫就會突然跳變，額外制造許多頻率成分。像逐漸淡入淡出音量，我們可以將框邊緣慢慢壓小，減輕接縫影響。這叫加窗windowing。

前置是[12.3時間框](https://birdhackor.github.io/tiny-perceptron-vlm/12.3.html)。Hann窗是一串從邊緣接近0、中間接近1的權重，波形逐點乘上它。它減少頻譜泄漏，即本來集中在一個頻率附近的能量因截斷而散到別處；代價是主峰會變寬，所以不是讓所有頻率判斷無條件更準。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.multimodal import tone

frame = tone()[:400]
window = torch.hann_window(400)
weighted = frame * window
print("起點權重", window[0].item(), "中間權重", window[200].item())
print("最後權重", round(window[-1].item(), 6))
print("原能量", round(frame.square().sum().item(), 2))
print("加窗能量", round(weighted.square().sum().item(), 2))

輸入400點單音與同長窗。`square().sum()`把每點振幅平方再加總，這裡當作框的能量量度。輸出起點0、中間1、最後約0.000062，能量50降到18.75。因為窗壓小了很多樣本，這個降低是預期結果，不是聲音憑空消失或模型訓練變好。

PyTorch預設periodic Hann，公式使用N個取樣點作為週期長度，所以最後一點接近0而不必恰好0；另一種對稱窗把兩端都設為0，公式分母會用N−1。若比較不同工具的頻譜，要記錄用哪一種，不能把末端微小差值誤判為程式錯誤。這裡不必背餘弦公式，重點是逐點權重與邊緣處理的目的。

加窗沒有改變取樣率、框長或hop，也沒有產生新的時間軸。它改變的是同一框內哪些點影響頻率統計較大。靜音區域、重取樣與錄音長度仍需各自處理，不能期待一個窗函數統一解決所有音訊問題。

接縫問題可以先從波形看：假如框末振幅0.4、框首-0.3，首尾直接相接時一下跳0.7，這不是原本連續聲音中的變化，卻會被頻率分析解釋成額外快起伏。窗把兩端壓向0，讓這個人工跳變減小。它同時也弱化框邊緣真實內容，所以相鄰重疊框有價值：一段聲音在某框邊緣被壓低，可能在下一框中心得到較大權重。窗與hop因此是相關的前處理選擇，但仍應分別記錄，不能只寫「做了頻譜」就丟掉它們的具體設定。

練習只把`torch.hann_window(400)`換成`torch.ones(400)`，先預測起點、中間、最後都1，加窗能量等於原能量50，再執行核對。這叫矩形窗，等於保留整框不淡入淡出；下一節的頻率分析便能比較不同邊緣處理造成的影響。
